# add-cxr-to-multimodal-task
---
## Setup

In [1]:
%%capture

from datetime import datetime
from typing import Any, Dict, List, Optional
import os
import polars as pl

# PyHealth Packages
from pyhealth.datasets import MIMIC4Dataset
from pyhealth.tasks.multimodal_mimic4 import ClinicalNotesMIMIC4, ClinicalNotesICDLabsMIMIC4, ClinicalNotesICDLabsCXRMIMIC4
from pyhealth.tasks.base_task import BaseTask

# Load MIMIC4 Files
# There's probably better ways dealing with this on the cluster, but working locally for now 
# (see: https://github.com/sunlabuiuc/PyHealth/blob/master/examples/mortality_prediction/multimodal_mimic4_minimal.py)

TASK = "ClinicalNotesICDLabsCXRMIMIC4" # The idea here is that we want additive tasks so we can evaluate the value in adding more modalities

PYHEALTH_REPO_ROOT = '/Users/wpang/Desktop/PyHealth'

EHR_ROOT = os.path.join(PYHEALTH_REPO_ROOT, "local_data/local/data/physionet.org/files/mimiciv/2.2")
NOTE_ROOT = os.path.join(PYHEALTH_REPO_ROOT, "local_data/local/data/physionet.org/files/mimic-iv-note/2.2")
CXR_ROOT = os.path.join(PYHEALTH_REPO_ROOT,"local_data/local/data/physionet.org/files/mimic-cxr-jpg/2.0.0")
CACHE_DIR = os.path.join(PYHEALTH_REPO_ROOT,"local_data/local/data/wp/pyhealth_cache")

DEV_MODE = True

if __name__ == "__main__":

    if TASK == "ClinicalNotesMIMIC4": # A bit janky setup at the moment and open to iteration, but conveys the point for now
        dataset = MIMIC4Dataset(
                ehr_root=EHR_ROOT,
                note_root=NOTE_ROOT,
                ehr_tables=["diagnoses_icd", "procedures_icd", "prescriptions", "labevents"],
                note_tables=["discharge", "radiology"],
                cache_dir=CACHE_DIR,
                num_workers=8,
                dev=DEV_MODE
            )
    
    elif TASK == 'ClinicalNotesICDLabsMIMIC4':
        dataset = MIMIC4Dataset(
                ehr_root=EHR_ROOT,
                note_root=NOTE_ROOT,
                ehr_tables=["diagnoses_icd", "procedures_icd", "prescriptions", "labevents"],
                note_tables=["discharge", "radiology"],
                cache_dir=CACHE_DIR,
                num_workers=8,
                dev=DEV_MODE
            )

    elif TASK == 'ClinicalNotesICDLabsCXRMIMIC4':
        dataset = MIMIC4Dataset(
                ehr_root=EHR_ROOT,
                note_root=NOTE_ROOT,
                cxr_root=CXR_ROOT,
                ehr_tables=["diagnoses_icd", "procedures_icd", "prescriptions", "labevents"],
                note_tables=["discharge", "radiology"],
                cxr_tables=["metadata", "negbio"],
                cache_dir=CACHE_DIR,
                num_workers=8,
                dev=DEV_MODE
            )

In [2]:
# negbio_patient_ids = (                                                                                                
#       dataset.global_event_df                                                                                           
#       .filter(pl.col("event_type") == "negbio")                                                                         
#       .select("patient_id")                                                                                             
#       .unique()                                                                                                         
#       .collect(engine="streaming")
#       .to_series()                                                                                                      
#       .to_list()                                                                                                        
#   )  
# negbio_patient_ids

# OR 
# dataset.unique_patient_ids[2]

ID = "17475607" # 17475607, 19689677, 11063609, 13208940
print(f"Patient ID is {ID}")

Patient ID is 17475607


In [3]:
# Single patient
patient = dataset.get_patient(ID)  

### Admission to Process

In [4]:
%%capture
admissions = patient.get_events(event_type="admissions")
# Determine which admissions to process iteratively
# Check each admission's NEXT admission for mortality flag
admissions_to_process = []
mortality_label = 0

for i, admission in enumerate(admissions):
    # Check if THIS admission has the death flag
    if admission.hospital_expire_flag in [1, "1"]:
        # Patient died in this admission - set mortality label
        # but don't include this admission's data
        mortality_label = 1
        break

    # Check if there's a next admission with death flag
    if i + 1 < len(admissions):
        next_admission = admissions[i + 1]
        if next_admission.hospital_expire_flag in [1, "1"]:
            # Next admission has death - include current, set mortality
            admissions_to_process.append(admission)
            mortality_label = 1
            break

    # No death in current or next - include this admission
    admissions_to_process.append(admission)

In [5]:
# admissions_to_process

### XRay

#### Negbio Events

In [6]:
%%capture
negbio_events = patient.get_events(event_type="negbio")
print(len(negbio_events))
print(negbio_events)

In [7]:
NEGBIO_FINDING_NAMES = [
            "no finding",
            "enlarged cardiomediastinum",
            "cardiomegaly",
            "lung opacity",
            "lung lesion",
            "edema",
            "consolidation",
            "pneumonia",
            "atelectasis",
            "pneumothorax",
            "pleural effusion",
            "pleural other",
            "fracture",
            "support devices"
    ]

all_negbio_findings = []
TOKEN_REPRESENTING_MISSING_TEXT = ""

In [8]:
for cxr in negbio_events: # Loop through each CXR
            negbio_vector = [] # Per CXR Vector
            try:
                for finding_name in NEGBIO_FINDING_NAMES: # Check each CXR's NEGBIO_FINDING_NAMES
                    try:
                        negbio_value = getattr(cxr, finding_name, TOKEN_REPRESENTING_MISSING_TEXT) 
                        if negbio_value!= TOKEN_REPRESENTING_MISSING_TEXT and float(negbio_value) > 0:
                            negbio_vector.append(finding_name)
                    except (ValueError, TypeError, AttributeError):
                        negbio_vector.append(TOKEN_REPRESENTING_MISSING_TEXT)
            except Exception: # Missing negbio for a given cxr
                negbio_vector = [TOKEN_REPRESENTING_MISSING_TEXT] * len(NEGBIO_FINDING_NAMES)

            all_negbio_findings.append(negbio_vector)

In [9]:
all_negbio_findings

[['', '', '', '', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', '', '', '', ''],
 ['no finding', '', '', '', '', '', '', '', '', '', '', '', '', ''],
 ['no finding', '', '', '', '', '', '', '', '', '', '', '', '', ''],
 ['no finding', '', '', '', '', '', '', '', '', '', '', '', '', ''],
 ['no finding', '', '', '', '', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', '', '', '', ''],
 ['no finding', '', '', '', '', '', '', '', '', '', '', '', ''],
 ['no finding', '', '', '', '', '', '', '', '', '', '', '', ''],
 ['no finding', '', '', '', '', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', ''],
 ['', '', '', '', '', '', '', '', '', ''],
 ['', '', '', 'lung opacity', '', '', '', '', '', '', '', ''],
 ['', '', '', 'lung opacity', '', '', '', '', '', '', 

#### Metadata

In [10]:
metadata_events = patient.get_events(event_type="metadata")
# print(len(metadata_events))
# print(metadata_events)

In [11]:
all_cxr_image_paths = []
all_cxr_hours_relative_to_nearest_admission = []

In [12]:
for cxr in metadata_events: # Loop through each CXR
    try:
        if cxr.image_path:
            cxr_image_path = cxr.image_path
            cxr_image_timestamp = cxr.timestamp

            # TODO: Consider making this into a utility function
            if cxr_image_timestamp is not None and admissions_to_process:
                nearest_admission = min(
                    admissions_to_process,
                    key=lambda a: abs((cxr_image_timestamp - a.timestamp).total_seconds()),
                )
                image_hours_from_nearest_admission = (
                    (cxr_image_timestamp - nearest_admission.timestamp).total_seconds() / 3600.0
                )

            all_cxr_image_paths.append(cxr_image_path)
            all_cxr_hours_relative_to_nearest_admission.append(image_hours_from_nearest_admission)
    except AttributeError:
        all_cxr_image_paths.append(self.TOKEN_REPRESENTING_MISSING_TEXT)
        all_cxr_hours_relative_to_nearest_admission(self.TOKEN_REPRESENTING_MISSING_FLOAT)

In [16]:
all_cxr_hours_relative_to_nearest_admission

[-6.3244444444444445,
 -6.3244444444444445,
 -3.8469444444444445,
 -1.0361111111111112,
 -1.0361111111111112,
 -1.0361111111111112,
 -2567.913611111111,
 -2567.913611111111,
 -2040.1083333333333,
 -2040.1083333333333,
 -484.65527777777777,
 -484.65527777777777,
 -1.0986111111111112,
 -80.15055555555556,
 -80.15055555555556,
 -2.2516666666666665,
 -2.2516666666666665,
 -2.2516666666666665,
 -3.5947222222222224,
 -3.5947222222222224,
 -1.8402777777777777,
 -1.8402777777777777,
 -5.815277777777778,
 -5.815277777777778,
 -3.162777777777778,
 -3.162777777777778,
 -3.162777777777778,
 -4.3436111111111115,
 -4.3436111111111115,
 -4.3436111111111115,
 -7.699166666666667,
 -7.699166666666667,
 30.808055555555555,
 48.74055555555555,
 48.74055555555555,
 76.07472222222222,
 97.49111111111111,
 120.65583333333333,
 224.35,
 241.7463888888889,
 -2.8491666666666666,
 -2.8491666666666666,
 -2.6397222222222223,
 -2.6397222222222223,
 13.1975,
 -10.793611111111112,
 -10.793611111111112,
 -4.8730555555

### Run Task 

In [13]:
# # Apply multimodal task
# task = ClinicalNotesICDLabsCXRMIMIC4() 
# samples = task(patient)

In [14]:
# samples